<center>
    <p>116 Advanced Statistics</p>
    <h1></h1>
    <h1>Lecture 03:</h1>
    <h1>Pairwise association and linear least squares</h1>
    <p>----</p>
    <p>Prof. Jochen Braun Ph.D.</p>
    <p>Institute of Biology</p>
    <p>Otto-von-Guericke University Magdeburg</p>
    <p>----</p>
    <p>Textbook:</p>
    <p> Press, Teukolsky, Vetterling, Flannery (2002)</p>   
    <p> Numerical Recipes in C++, Cambridge University Press.</p>
    <p> B Efron & T Hastie (2016) Computer Age Statistical Inference,  </p>
    <p> Cambridge University Press.</p>
</center>

# **Abstract:**

We meet different ways of **testing statistical associations** between both categorical and continuous samples. Futher, we introduce the difference between **parametric** tests (which assume normal distributions) and **non-parametric** tests (which don't), including **information-based measures**.

Then we enter the realm of **maximum likelihood** fitting or **linear regression**, working out the linear algebra of least squares methods and illustrating them with several examples.




# **Overview:**

**1. Parametric measures**

**2. Non-parametric measures**

**3. Information-based measures**

**4. Maximum likelihood and modeling of associations**

**5. Fitting lines (linear least squares)**

**6. Fitting curves (general linear least squares)**







# **1. Parametric measures of association**

A common situation is that data points have several different quantities (two or more) associated with them and we wish to know to what extent one quantity can predict the other? For example, this happens when we measure several different quantities in individual animals or human subjects.

In such situations, we typically have two questions. Is there a significant association between the two quantities and, if yes, how strong is this assocation?

We start with some classical 'parametric' methods:

- **chi-square measure of association** for **categorical** variables

-  **linear correlation coefficients** for **continuous** variables



# Measure of association based on $\chi^2$

The association between two categorical variables $x$ and $y$ can always be turned into a contingency table. Here we choose $rows$ for variable $x$, indexed by $i \in \{1, 2, \ldots , I\}$ and $columns$ for variable $y$, indexed by $j\in\{1, 2, \ldots , J\}$:

<br>

$$\begin{eqnarray}
\begin{array}{|c|ccc|c|}
\hline  
& \mathrm{Col 1} & \mathrm{Col 2} & \ldots & y \\
\hline
\mathrm{Row 1} & n_{11} & n_{12}& \ldots & N_{(1\mathbin{\bullet})}\\
\mathrm{Row 2} & n_{21} & n_{22}& \ldots & N_{(2\mathbin{\bullet})}\\
\vdots & \vdots & \vdots & \ddots & \vdots \\
\hline
x & N_{(\mathbin{\bullet}1)} & N_{(\mathbin{\bullet}2)} & \ldots & N \\
\hline
\end{array}
\end{eqnarray}$$

<br>

Summing all entries in a row gives us the **row sums**

$$
\sum_j n_{ij} = N_{(i\mathbin{\bullet})}
$$

and summing all entries in a column gives us the **column sums**

$$
\sum_i n_{ij} = N_{(\mathbin{\bullet}j)}
$$

Naturally, both row sums and column sums add up to the total number of observations, $N$

$$
\sum_i N_{(i\mathbin{\bullet})} = \sum_j N_{(\mathbin{\bullet}j)} = N
$$

Under the null hypothesis that both categories are independent, the expected number or entries, which we denote $e_{ij}$, follows from the row and column sums

$$
\frac{e_{ij}}{N_{(\mathbin{\bullet}j)}} = \frac{N_{(i\mathbin{\bullet})}}{N} \qquad \Leftrightarrow \qquad e_{ij} = \frac{N_{(i\mathbin{\bullet})} \, N_{(\mathbin{\bullet}j)}}{N}
$$

<br>

The chi-square statistic sums over both rows and columns

$$
\chi^2 = \sum_{i,j} \frac{\left(n_{ij} - e_{ij} \right)^2}{e_{ij}}
$$

<br>

The number of degrees of freedom is the number of entries in the table, minus the number of constraints used to compute $e_{ij}$. Each row total and column total is a constraint, except that this overcounts by one, sind row and column totals add up to $N$. Accordingly

<br>

$$
\mathrm{dof} = (I-1)(J-1) = I J - I - J + 1
$$

<br>

We can now use the $\chi^2$ test for a significant assocation: gamma distribution with shape parameter $dof/2$ and scale parameter $\chi^2/2$.

<br>

An old-fashioned measure for the **strength** of such an association is **Cramer's V**:

$$
V = \sqrt{\frac{\chi^2}{N \min(I-1, J-1)}}
$$

This measure ranges from zero (no association) to unity (perfect association).  **A perfect association means that each row and each column have exactly one entry.**


In [ ]:
from inspect import signature
# @title Chi-square measure of association {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import norm
from scipy.stats import gamma
from matplotlib.colors import LinearSegmentedColormap


def ThreeObservations(N):

    Nrange = 20;

    Y1  = np.random.uniform(0,Nrange+1,N);    # uniformly distributed between 1 and 20
    Y1  = np.floor( Y1 ).astype(int);

    Off1 = (4.0*(Y1-Nrange/2)/5.0);        # offset depends linearly on R1

    Mu2 = (Nrange/2 + Off1) / Nrange;
    N2  = np.ones(N,dtype='int') * Nrange;
    Y2  = np.random.binomial(N2, Mu2);       # binomially distributed around mean R1

    RandomSign = 2*np.round(np.random.uniform(0,1,N))-1;
    Mu3 = (Nrange/2 +  Off1 * RandomSign) / Nrange;
    Y3  =  np.random.binomial( N2, Mu3);       # bimodally and binomially distributed around mean +-R1

    return Y1, Y2, Y3;

def ChisquareAssociation( x, y ):

    Nrowx = np.max(x) + 1;
    Ncoly = np.max(y) + 1;

    Nxy = np.zeros((Nrowx, Ncoly), dtype=int)
    np.add.at(Nxy, (x, y), 1)

    #total counts
    N = np.sum(Nxy);

    # sum over columns, retain rows
    Nx = np.sum(Nxy, axis=0);
    Nx = np.tile( Nx, (Nrowx, 1))

    # sum over rows, retain columns
    Ny = np.sum(Nxy, axis=1);
    Ny = np.tile( Ny, (Ncoly, 1)).T;

    # compute predictions
    nxy = Nx * Ny / N;

    # squared deviations from predictions
    chi2_unnorm = (Nxy - nxy) ** 2;

    # normalize
    ix = np.where( nxy != 0 )
    chi2norm = chi2_unnorm[ix] / nxy[ix];

    chi2stat = np.sum( chi2norm );

    return Nrowx, Ncoly, Nxy, nxy, chi2stat;



def plot_chisquare_association( log10N=4 ):
    """(Slide-50)"""

    # sample size
    N = round(10**log10N);

    # generate three pairs of neurons R
    (y1, y2, y3) = ThreeObservations( N );

    # compute contingency tables
    (Nrow1, Ncol2, Ny1y2, ny1y2, chi2stat12) = ChisquareAssociation( y1, y2 );

    p_value12 = 1-gamma.cdf( chi2stat12/2, (Nrow1-1)*(Ncol2-1)/2 );

    chi2perdof12   = chi2stat12 / ( (Nrow1-1)*(Ncol2-1 ) )

    kramersV12 = np.sqrt( chi2stat12 / (N*np.min([Nrow1-1, Ncol2-1])) )

    # visualize
    fig, axs = plt.subplots( 2,1, figsize=(8,12))

    axs[0].imshow( Ny1y2, cmap='viridis', aspect='equal' )
    axs[0].set_title('Observed association: Y1, Y2', fontsize=18)
    axs[0].set_xlabel('Y2')
    axs[0].set_ylabel('Y1')

    axs[0].text( 2, 10, f"P-value: {p_value12:.3f}", color='w', fontsize=18 )
    axs[0].text( 2, 12.5, f"Chi2/dof: {chi2perdof12:.3f}", color='w', fontsize=18 )
    axs[0].text( 2, 15, f"Kramers V: {kramersV12:.3f}", color='w', fontsize=18 )


    # compute contingency tables
    (Nrow1, Ncol3, Ny1y3, ny1y3, chi2stat13) = ChisquareAssociation( y1, y3 );

    p_value13 = 1-gamma.cdf( chi2stat13/2, (Nrow1-1)*(Ncol3-1)/2 );

    chi2perdof13   = chi2stat13 / ( (Nrow1-1)*(Ncol3-1 ) )

    kramersV13 = np.sqrt( chi2stat13 / (N*np.min([Nrow1-1, Ncol2-1])) )

    # visualize
    axs[1].imshow( Ny1y3, cmap='viridis', aspect='equal' )
    axs[1].set_title('Observed association: Y1, Y3', fontsize=18)
    axs[1].set_xlabel('Y3')
    axs[1].set_ylabel('Y1')

    axs[1].text( 2, 10, f"P-value: {p_value13:.3f}", color='w', fontsize=18 )
    axs[1].text( 2, 12.5, f"Chi2/dof: {chi2perdof13:.3f}", color='w', fontsize=18 )
    axs[1].text( 2, 15, f"Kramers V: {kramersV13:.3f}", color='w', fontsize=18 )


    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_chisquare_association, log10N = (1.0, 7.0, 1 ) )

interactive(children=(FloatSlider(value=4.0, description='log10N', max=7.0, min=1.0, step=1.0), Output()), _do…

<function __main__.plot_chisquare_association(log10N=4)>

# Linear correlation coefficient


The most widely used measure is the **linear correlation coefficient**, also known as **Pearson's r**.

Given paired observations $(x_i, y_i)$ with $i=1,2,\ldots, N$, this is defined as

$$
r \equiv \frac{\sum_i (x_i - \mu_x)(y_i - \mu_y)}{\sqrt{\sum_i (x_i-\mu_x)^2 }\sqrt{\sum_i (y_i-\mu_y)^2}}
$$

<br>

Actually, this looks far more complicated than it really is. (Psychologists do love to bamboozle you with needless complexity.) In fact, the formulat is equivalent to the **expected product of the z-scored variables:**

$$\begin{eqnarray}
r&=&\frac{\frac{1}{N}\sum_i (x_i - \mu_x)(y_i - \mu_y)}{\sqrt{\frac{1}{N}\sum_i (x_i-\mu_x)^2 }\sqrt{\frac{1}{N}\sum_i (y_i-\mu_y)^2}} =
\\
&=& \frac{\frac{1}{N}\sum_i (x_i - \mu_x)(y_i - \mu_y)}{\sigma_x \sigma_y}=
\\
&=&\frac{1}{N} \sum_i \frac{x_i-\mu_i}{\sigma_x} \frac{y_i-\mu_y}{\sigma_y} = \frac{1}{N} \sum_i z(x_i) z(y_i)
\end{eqnarray}$$

<br>

$$
r = \langle z(x) \cdot z(y)\rangle, \qquad\quad z(x) \equiv \frac{x_i - \mu_x }{\sigma_x}, \qquad\quad z(y) \equiv \frac{y_i - \mu_y}{\sigma_y}
$$

<br>

with the means and variances of $x$ and $y$ computed as

$$
\mu_x = \frac{1}{N} \sum_i x_i, \qquad \mu_y = \frac{1}{N} \sum_i y_i, \qquad \sigma_x^2 = \frac{1}{N} \sum_i (x-\mu_x)^2 ,\qquad \sigma_y^2 = \frac{1}{N} \sum_i (y-\mu_y)^2
$$

Bessel's correction is ignored because we deal with large $N$.

<br>

Unfortunately, correlation coefficients $r$ are **not particularly informative** about **statistical significance**. When the underlying observations $x$ and $y$ are approximately normally distributed, in other words, when

$$
z(x,y) \sim \mathscr{N}(0,1)
$$

we can transform $r$-values with Fisher's z-transform

$$
z_r = \frac{1}{2} \ln \left(\frac{1+r}{1-r} \right)
$$

because the $z_r$ values will be approximately normally distributed

$$
z_r \sim \mathscr{N}(\mu_z, \sigma_z)
$$

with mean and variance

$$
\mu_z = \frac{1}{2}\left[\ln\left(\frac{1+r_\mathrm{true}}{1-r_\mathrm{true}} \right) + \frac{r_\mathrm{true}}{N-1}\right]
,\qquad\qquad
\sigma_z^2 \approx \frac{1}{N-3}
$$

<br>

So the significance level of an observe (absolute) value of $\left|z_r\right|$ is

$$
p = \mathrm{erfc}\left( \frac{\left|z_r\right| \sqrt{N-3}}{\sqrt{2}} \right)
$$

and the significance of the difference between two observed values $z_1$ and $z_2$ is

$$
p = \mathrm{erfc}\left( \frac{\left|z_1-z_2\right|}{\sqrt{2} \sqrt{\frac{1}{N_1-3}+\frac{1}{N_2-3}}} \right)
$$

where $z_1$ and $z_2$ are obtained from $r_1$ and $r_2$, respectively, and $N_1$ and $N_2$ are the numbers of data points.

<br>

This approach can be used when assessing a **large number of pairwise correlations**, for example in functional MRI data.  However, the interpretation requires a **false discovery correction (FDR)**, as discussed in Lecture 5.



In [ ]:
from inspect import signature
# @title Linear correlation coefficient {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import norm
from scipy.stats import gamma
from scipy.special import erfc
from matplotlib.colors import LinearSegmentedColormap


def ThreeObservations(N):

    Nrange = 20;

    Y1  = np.random.uniform(0,Nrange+1,N);    # uniformly distributed between 1 and 20
    Y1  = np.floor( Y1 ).astype(int);

    Off1 = (4.0*(Y1-Nrange/2)/5.0);        # offset depends linearly on R1

    Mu2 = (Nrange/2 + Off1) / Nrange;
    N2  = np.ones(N,dtype='int') * Nrange;
    Y2  = np.random.binomial(N2, Mu2);       # binomially distributed around mean R1

    RandomSign = 2*np.round(np.random.uniform(0,1,N))-1;
    Mu3 = (Nrange/2 +  Off1 * RandomSign) / Nrange;
    Y3  =  np.random.binomial( N2, Mu3);       # bimodally and binomially distributed around mean +-R1

    return Y1, Y2, Y3;

def LinearCorrelationCoefficient( x, y ):

    # size of data set

    N = len(x);

    # linear correlation
    rho = np.corrcoef( x, y )

    #print( f"Correlation coefficient: {rho[0,1]:.3f}" )

    rhoxy = rho[0,1]

    # z transform

    zxy = 0.5 * np.log( (1+rhoxy)/(1-rhoxy) );

    pxy = erfc( np.abs(zxy) * np.sqrt(N-3) / np.sqrt(2) );

    #print( f"P-value: {pvalue:.3f}"

    return rhoxy, pxy;



def plot_linear_correlation( log10N=4 ):
    """(Slide-50)"""

    # sample size
    N = round(10**log10N);

    # generate three pairs of neurons R
    (y1, y2, y3) = ThreeObservations( N );

    # compute contingency tables
    (Nrow1, Ncol2, Ny1y2, ny1y2, chi2stat12) = ChisquareAssociation( y1, y2 );

    (rho12, p12) = LinearCorrelationCoefficient( y1, y2 );

    # visualize
    fig, axs = plt.subplots( 2,1, figsize=(8,12))

    axs[0].imshow( Ny1y2, cmap='viridis', aspect='equal' )
    axs[0].set_title('Observed association: Y1, Y2', fontsize=18)
    axs[0].set_xlabel('Y2')
    axs[0].set_ylabel('Y1')

    axs[0].text( 2, 10, f"Rho: {rho12:.3f}", color='w', fontsize=18 )
    axs[0].text( 12, 10, f"P-value: {p12:.3f}", color='w', fontsize=18 )


    # compute contingency tables
    (Nrow1, Ncol3, Ny1y3, ny1y3, chi2stat13) = ChisquareAssociation( y1, y3 );

    (rho13, p13) = LinearCorrelationCoefficient( y1, y3 );

    # visualize
    axs[1].imshow( Ny1y3, cmap='viridis', aspect='equal' )
    axs[1].set_title('Observed association: Y1, Y3', fontsize=18)
    axs[1].set_xlabel('Y3')
    axs[1].set_ylabel('Y1')

    axs[1].text( 2, 10, f"Rho: {rho13:.3f}", color='w', fontsize=18 )
    axs[1].text( 12, 10, f"P-value: {p13:.3f}", color='w', fontsize=18 )

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_linear_correlation, log10N = (1.0, 7.0, 1 ) )

interactive(children=(FloatSlider(value=4.0, description='log10N', max=7.0, min=1.0, step=1.0), Output()), _do…

<function __main__.plot_linear_correlation(log10N=4)>

# **2. Non-parametric measures of association**

The difficulty in assessing the significance of linear correlations leads us to consider **nonparametric** or **rank-order** correlations.
Once again we consider paired  observations $(x_i, y-i)$, but without relying on assumptions about the distributions from which they are drawn.

<br>

To obtain a rank-order correlation, we sort all values of $x_i$ (and, separately $y_i$) in ascending order and replace every numerical value by the associated rank, $1, 2, 3, \ldots, N$. The resulting list of numbers is now drawn from a perfectly known distribution, namely, uniformly from the list of ranks $1, 2, 3, \ldots, N$.

<br>

When some observations $x_i$ are identical, we assign all **'ties'** to the mean of the nominal ranks. Naturally, this midrank will sometimes be integer and sometimes half-integer. Note that this procedure ensures that the sum of all ranks equals $\frac{1}{2}N(N+1)$, the sum of all integers from $1$ to $N$.

<br>

Ranking incurs a small loss of information, but the advantage is considerable: when a non-parametric correlation is detected (at a chosen level of significance), then it is really there. Nonparametric correlation is more robust when the correlation is monotonic and nonlinear and it is also more resistent to outliers,
in the same sense that the median is more robust than the mean.




# Spearman Rank-Order correlation

Let $R_i$ be the rank of $x_i$ among the other $x_i$'s, and $S_i$ the rank of $y_i$ among the other $y_i$'s. Then the rank-order correlation is defined simply as

$$
r_s \equiv \frac{\sum_i (R_i - \mu_R)(S_i - \mu_S)}{\sqrt{\sum_i (R_i-\mu_R)^2 }\sqrt{\sum_i (S_i-\mu_S)^2}}
$$

with the means


$$
\mu_R = \frac{1}{N} \sum_i R_i, \qquad \mu_y = \frac{1}{N} \sum_i S_i
$$

The significance of a nonzero value of $r_s$ is obtained from a t-statistic with $N-2$ degrees of freedom

$$
t = r_s \sqrt{\frac{N-2}{1-r_s^2}}
$$

Another, more intutive measure is the *sum squared difference of ranks*, defined as

$$
D = \sum_{i=1}^N (R_i-S_i)^2
$$

The relation is

$$
r_s = 1 - \frac{6D}{N^3-N} \qquad \Leftrightarrow \qquad D = \frac{(1-r_s)(N^3-N)}{6}
$$

if there are no ties in the data.  In the presence of ties, the formula is more complicated.



In [ ]:
from inspect import signature
# @title Spearman Rank-Order Correlation {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import norm
from scipy.stats import gamma
from scipy.stats import rankdata
from scipy.stats import t
from scipy.special import erfc
from matplotlib.colors import LinearSegmentedColormap


def MakeObservations(N):       # N number of data pairs

    Nrange = 20;               # set scale

    SigmaY = 200;

    X  = np.random.uniform(0,Nrange+1,N);      # uniformly distributed between 0 and 20

    Ymax = 0.5 * Nrange**3;                    # upper bound

    MuY = 0.5 * X**3 - 0.5 * Ymax             # non-linear but monotonic dependence, between 0 and 200

    Y  = np.random.normal(MuY, SigmaY );       # binomially distributed around mean R1

    Noutlier = np.floor(N/5).astype(int)       #  add 20% outliers

    ix = np.random.permutation( np.arange(0,N));

    Y[ ix[0:Noutlier] ] = np.random.uniform(low=-Ymax, high=Ymax, size=Noutlier)

    return X, Y;

def ThreeObservations(N):

    Nrange = 20;

    Ysigma = 200;

    Y1  = np.random.uniform(0,Nrange+1,N);      # uniformly distributed between 0 and 20

    Y2max = 0.5 * Nrange**3;                    # upper bound

    Mu2 = 0.5 * Y1**3 - 0.5 * Y2max             # non-linear but monotonic dependence, between 0 and 200

    Y2  = np.random.normal(Mu2, Ysigma );       # binomially distributed around mean R1


    Y3  = np.random.normal(Mu2, Ysigma );       # binomially distributed around mean R1

    Nout = np.floor(N/5).astype(int)

    ix = np.random.permutation( np.arange(0,N));      #  add 10% outliers

    Y3[ ix[0:Nout] ] = np.random.uniform(low=-Y2max, high=Y2max, size=Nout)

    return Y1, Y2, Y3;


def LinearCorrelationCoefficient( x, y ):

    # size of data set

    N = len(x);

    # linear correlation
    rhoxy = np.corrcoef( x, y )[0,1]

    # z transform

    zxy = 0.5 * np.log( (1+rhoxy)/(1-rhoxy) );

    pxy = erfc( np.abs(zxy) * np.sqrt(N-3) / np.sqrt(2) );

    #print( f"P-value: {pvalue:.3f}"

    return rhoxy, pxy;

def RankOrderCorrelation( x, y ):

    # size of data set

    N = len(x);

    # Rank the data
    rank_x = rankdata(x)
    rank_y = rankdata(y)

    # Pearson correlation on the ranks
    rhospear = np.corrcoef(rank_x, rank_y)[0, 1]

    # t-statistic
    tspear = np.sqrt( (N-2) * rhospear**2 / (1-rhospear**2))

    pspear= t.sf( np.abs(tspear), N-2 );

    return rhospear, tspear, pspear

def plot_spearman_correlation( log10N=2 ):
    """(Slide-50)"""

    # fontsize
    fs = 18;

    # sample size
    N = round(10**log10N);

    # generate three pairs of neurons R
    (y1, y2, y3) = ThreeObservations( N );

    # compute contingency tables
    (rho12, p12) = LinearCorrelationCoefficient( y1, y2 );

    (rspear12, tspear12, pspear12) = RankOrderCorrelation( y1, y2 );

    # visualize
    fig, axs = plt.subplots( 2,1, figsize=(8,12))

    axs[0].plot( y1, y2, 'ko', markersize=2 )
    axs[0].set_title('Observed association: Y1, Y2', fontsize=fs)
    axs[0].set_xlabel('Y2')
    axs[0].set_ylabel('Y1')
    axs[0].set_box_aspect(1/1)

    axs[0].text( 2, -200, f"Rho: {rho12:.3f}", color='b', fontsize=fs )
    axs[0].text( 12, -200, f"P-value: {p12:.3f}", color='b', fontsize=fs )
    axs[0].text( 2, +200, f"Rho Spear : {rspear12:.3f}", color='r', fontsize=fs )
    axs[0].text( 12, +200, f"P-value: {pspear12:.3f}", color='r', fontsize=fs )

    (rho13, p13) = LinearCorrelationCoefficient( y1, y3 );
    (rspear13, tspear13, pspear13) = RankOrderCorrelation( y1, y3 );


    # visualize
    axs[1].plot( y1, y3, 'ko', markersize=2 )
    axs[1].set_title('Observed association: Y1, Y3', fontsize=fs)
    axs[1].set_xlabel('Y3')
    axs[1].set_ylabel('Y1')
    axs[1].set_box_aspect(1/1)

    axs[1].text( 2, -200, f"Rho: {rho13:.3f}", color='b', fontsize=fs )
    axs[1].text( 12, -200, f"P-value: {p13:.3f}", color='b', fontsize=fs )
    axs[1].text( 2, +200, f"Rho Spear: {rspear13:.3f}", color='r', fontsize=fs )
    axs[1].text( 12, +200, f"P-value: {pspear13:.3f}", color='r', fontsize=fs )

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_spearman_correlation, log10N = (1.0, 7.0, 1 ) )

interactive(children=(FloatSlider(value=2.0, description='log10N', max=7.0, min=1.0, step=1.0), Output()), _do…

<function __main__.plot_spearman_correlation(log10N=2)>

# Kendall's Tau

An even more non-parametric test is Kendall's Tau, which uses only relative rank: higher, lower, or equal.  In most applications, rank order $r_s$ and $tau$ produce the same result.

To define $\tau$ we start with $N$ data points $(x_i,y_i)$ consider all $\frac{1}{2} N (N-1)$ *pairs* of data points $i$, $j$.  A pair is called *concordant* if the relative ordering of ranks (or underlying observations) is the same. A pair is called *discordant* if the relative ordering is the opposite. A pair with one tie and one ordering is called an *extra-x* or *extra-y*, depending on which observation is ordered. A double tie is disregarded.

Kendall's $\tau$ simply compares these counts, as follows

$$
\tau = \frac{\mathrm{conc} - \mathrm{disc}}{\sqrt{\mathrm{conc}+\mathrm{disc}+\mathrm{extra_x}}\sqrt{\mathrm{conc}+\mathrm{disc}+\mathrm{extra_x}}}
$$

The expected value is zero and the variance (derived from combinatorics) is

$$
\sigma_\tau^2 = \frac{4N+10}{9N(N-1)}
$$

Beware that Kendall's Tau is an $O(N^2)$ algorithm and thus computes much more slowly than Rank-Order, which is $O(N\ln N)$.


# Wilcoxon rank-sum test for medians

For the sake of completeness, we mention another non-parametric test based on ranking, which serves to compare medians when the observations are not normally distributed (which was the theme of the last lecture).

Given two sets of observations $x_i$ and $y_i$ with $n_1$ and $n_2$ data points, respectively, we sort both sets together, assigning ranks from $1$ to $n_1+n_2$.

After forming the separate 'rank sums' of both sets, $w_1$ and $w_2$, we compute the test statistics

$$
u_1 = w_1 - \frac{n_1(n_1-1)}{2}, \qquad\qquad u_2 = w_2 - \frac{n_2(n_2-1)}{2}
$$

The smaller of the two test statistics $u = \min(u_1, u_2)$ is normally distribued

$$
z = \frac{u-\frac{n_1 n_2}{2}}{\sqrt{\frac{n_1 n_2(n_1+n_2+1)}{12}}} \sim \mathscr{N}(0,1)
$$

# **3. Information-based measures of association**

A more modern measure for the **strength** of an association is provided by **Shannon information or entropy**

$$
H = - \sum_i p_i \ln p_i
$$

To apply this approach, we convert our counts into probabilities

$$
p_{ij} = \frac{n_{ij}}{N}, \qquad p_{(i\mathbin{\bullet})} = \frac{N_{(i\mathbin{\bullet})}}{N}, \qquad p_{(\mathbin{\bullet}j)} = \frac{N_{(\mathbin{\bullet}j)}}{N}
$$

The entropies of the individual categories (marginal entropies) $x$ and $y$ are

$$
H(x) = -\sum_i p_{(i\mathbin{\bullet})} \ln p_{(i\mathbin{\bullet})} \,\, \, \qquad\qquad H(y) = -\sum_i p_{(\mathbin{\bullet}j)} \ln p_{(\mathbin{\bullet}j)}
$$

and the combined entropy of both categories (joint entropy) is

$$
H(x,y) = -\sum_{i,j} p_{ij} \ln p_{ij}
$$

and the conditional entropies (remaining uncertainty about one category if we know the other) are

$$
H(x|y) = \sum_i p_{(i\mathbin{\bullet})} \left[-\sum_j \frac{p_{ij}}{p_{(i\mathbin{\bullet})}} \ln \frac{p_{ij}}{p_{(i\mathbin{\bullet})}}\right]= -\sum_{i,j} p_{ij} \ln \frac{p_{ij}}{p_{(i\mathbin{\bullet})}}
$$

$$
H(y|x) = \sum_j p_{(\mathbin{\bullet}j)} \left[-\sum_j \frac{p_{ij}}{p_{(\mathbin{\bullet}j)}} \ln \frac{p_{ij}}{p_{(\mathbin{\bullet}j)}}\right]= -\sum_{i,j} p_{ij} \ln \frac{p_{ij}}{p_{(\mathbin{\bullet}j)}}
$$

We can now express the dependency of $y$ on $x$, in terms of the **uncertainty coefficient** of $y$:

$$
U(y|x) \equiv \frac{H(y) - H(y|x)}{H(y)}
$$

which measures the fractional entropy of $y$ that **does depend** on $x$. This is zero when $x$ provides no information about $y$ and unity when $x$ completely determines $y$ (i.e., a perfect association as defined above).

<br>

Similarly, we can express the dependency of $x$ on $y$, in terms of the **uncertainty coefficient** of $x$:

$$
U(x|y) \equiv \frac{H(x) - H(x|y)}{H(x)}
$$

which measures the fractional entropy of $x$ that **does depend** on $y$. This is zero when $y$ provides no information about $x$ and unity when $y$ completely determines $x$.

<br>

If we want to treat both categories $x$ and $y$ symmetrically, we can use

$$
U(x,y) \equiv 2 \, \frac{H(x) +H(y) - H(x,y)}{H(x)+H(y)}
$$

which is the mutual information as a fraction of the marginal entropies. This measure is zero if $x$ and $y$ are independent and unity if they are perfectly dependent.







In [ ]:
from inspect import signature
# @title Information-based measures {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import norm
from scipy.stats import gamma
from scipy.special import erfc
from matplotlib.colors import LinearSegmentedColormap


def ThreeObservations(N):

    Nrange = 20;

    Y1  = np.random.uniform(0,Nrange+1,N);    # uniformly distributed between 1 and 20
    Y1  = np.floor( Y1 ).astype(int);

    Off1 = (4.0*(Y1-Nrange/2)/5.0);        # offset depends linearly on R1

    Mu2 = (Nrange/2 + Off1) / Nrange;
    N2  = np.ones(N,dtype='int') * Nrange;
    Y2  = np.random.binomial(N2, Mu2);       # binomially distributed around mean R1

    RandomSign = 2*np.round(np.random.uniform(0,1,N))-1;
    Mu3 = (Nrange/2 +  Off1 * RandomSign) / Nrange;
    Y3  =  np.random.binomial( N2, Mu3);       # bimodally and binomially distributed around mean +-R1

    return Y1, Y2, Y3;

def UncertaintyCoefficients( x, y ):

    # size of data set

    Nrowx = np.max(x) + 1;
    Ncoly = np.max(y) + 1;

    Nxy = np.zeros((Nrowx, Ncoly), dtype=int)
    np.add.at(Nxy, (x, y), 1)

    #total counts
    N = np.sum(Nxy);

    #joint probability
    Pxy = Nxy / N;

    # sum over x columns, retain y rows
    Py = np.sum(Pxy, axis=0);
    Pyy = np.tile( Py, (Nrowx, 1))

    # sum over y rows, retain x columns
    Px = np.sum(Pxy, axis=1);
    Pxx = np.tile( Px, (Ncoly, 1)).T;

    # marginal entropies
    ix = np.where( Px != 0 )
    Hx = -np.sum( Px[ix] * np.log( Px[ix] ) )

    ix = np.where( Py != 0 )
    Hy = -np.sum( Py[ix] * np.log( Py[ix] ) )

    # joint entropy
    ix = np.where( Pxy != 0 )
    Hxy = -np.sum( Pxy[ix] * np.log( Pxy[ix] ) )

    # conditional entropies
    ix = np.where( Pxy != 0 )
    Hx_given_y = -np.sum( Pxy[ix] * np.log( Pxy[ix] / Pyy[ix] ) )

    ix = np.where( Pxy != 0 )
    Hy_given_x = -np.sum( Pxy[ix] * np.log( Pxy[ix] / Pxx[ix] ) )

    # uncertainty coefficients
    Ux_given_y = (Hy - Hx_given_y) / Hy;
    Uy_given_x = (Hx - Hy_given_x) / Hx;
    Uxy = 2 * (Hx + Hy - Hxy) / (Hx + Hy);

    return Ux_given_y, Uy_given_x, Uxy



def plot_uncertainty_coefficients( log10N=4 ):
    """(Slide-50)"""

    # sample size
    N = round(10**log10N);

    # generate three pairs of neurons R
    (y1, y2, y3) = ThreeObservations( N );

    # compute contingency tables
    (Nrow1, Ncol2, Ny1y2, ny1y2, chi2stat12) = ChisquareAssociation( y1, y2 );
    (Nrow1, Ncol3, Ny1y3, ny1y3, chi2stat13) = ChisquareAssociation( y1, y3 );


    # uncertainty coefficients
    (U1_given_2, U2_given_1, U12) = UncertaintyCoefficients( y1, y2 );
    (U1_given_3, U3_given_1, U13) = UncertaintyCoefficients( y1, y3 );


    # visualize
    fig, axs = plt.subplots( 2,1, figsize=(8,12))

    axs[0].imshow( Ny1y2, cmap='viridis', aspect='equal' )
    axs[0].set_title('Observed association: Y1, Y2', fontsize=18 )
    axs[0].set_xlabel('Y2')
    axs[0].set_ylabel('Y1')

    axs[0].text( 2, 10, f"U(1|2): {U1_given_2:.3f}", color='w', fontsize=18 )
    axs[0].text( 12, 10, f"U(2|1): {U2_given_1:.3f}", color='w', fontsize=18 )
    axs[0].text( 5, 5, f"U(1,2): {U12:.3f}", color='w', fontsize=18 )

    # visualize
    axs[1].imshow( Ny1y3, cmap='viridis', aspect='equal' )
    axs[1].set_title('Observed association: Y1, Y3', fontsize=18)
    axs[1].set_xlabel('Y3')
    axs[1].set_ylabel('Y1')

    axs[1].text( 2, 10, f"U(1|3): {U1_given_3:.3f}", color='w', fontsize=18 )
    axs[1].text( 12, 10, f"U(3|1): {U3_given_1:.3f}", color='w', fontsize=18 )
    axs[1].text( 5, 5, f"U(1,3): {U13:.3f}", color='w', fontsize=18 )

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_uncertainty_coefficients, log10N = (1.0, 7.0, 1 ) )

interactive(children=(FloatSlider(value=4.0, description='log10N', max=7.0, min=1.0, step=1.0), Output()), _do…

<function __main__.plot_uncertainty_coefficients(log10N=4)>

# **4. Maximum-likelihood and modeling of associations**

When we want to quantify systematic associations in our observed data, we have three large goals

<br>

- fit the parameters of a given model to observations

<br>

- estimate the errors on the fitted parameter values

<br>

- find a statistical measure for goodness of fit

<br>

If we could achieve these goals, we would have some rational basis for choosing between different models.

# Least squares and maximum likelihood

Suppose we are fitting $N$ data points $(x_i,y_i)$, where $i=1,2,\ldots, N$, to a model with $M$ adjustable parameters $a_j$, where $j=1,2,\ldots, M$. The model predicts the relation between independent variable $x$ and dependent variable $y$,

$$
y(x) = y(x; a_1, \ldots, a_M)
$$

<br>

How can we go about choosing the parameters $a_j$?

<br>

An utopian ideal would be the find the most likely model, given our observations. However, this would require a parametrized space containing all possible models, which does not exist.

<br>

Instead, let's ask ourselves how probable our observations $y_i$ would have been, given one specific model and  *particular set of parameters*? Of course, if the $y_i$ are continuous, the probability of reproducing our exact observations is zero, so we have to allow for some tolerance $\Delta y$.

<br>

Next, we interpret the *probability* of our observations, given the parameters, (which we can compute) as the *likelihood* of parameters, given our observations. If we are ready to accept this, then it is only a small step to fit parameters by finding the values that *maximize* the likelihood of observations.  This is called m*aximum likelihood estimation*.

<br>



# Maximum likelihood estimation

To make this more concrete, we assume that our measurement errors on the $y_i$ are independent and normally distributed with uniform variance $\sigma^2$.  Further below, we will modify these assumptions.

<br>

The probability of observations $\{x_1, x_2, \ldots, x_N\}$, given our model $y(x) = y(x; a_1, \ldots, a_M)$ and its parameters $\{a_1, \ldots, a_M\}$ is then

$$
P\propto \prod_{i=1}^N \left\{ \exp\left[-\frac{1}{2} \left(\frac{y_i - y(x_i)}{\sigma} \right)^2 \right] \Delta y \right\}
$$

<br>

Maximizing this is equivalent to **minimizing** the **negative** logarithm, namely

$$
-log P \propto \left[ \sum_{i=1}^N \frac{\left[y_i - y(x_i) \right]^2}{2\sigma^2} \right] - N\ln \Delta y
$$

<br>

Since $N$, $\Delta y$ and $\sigma$ are constants, this is equivalent to minimizing the sum of squared errors

$$
\mathrm{SSE} = \sum_{i=1}^N \left[y_i - y(x_i) \right]^2
$$

So we see that minimizing the SSE is equivalent to maximum likelihood estimation, if (**and this is a BIG if**)  measurement errors are distributed normally.



# Generalizations

We can generalize this by allowing for non-uniform variances $\sigma_i^2$ of measurement errors, assuming that we have some observations on these values, by substituting

$$
P(y_i|x_i) \propto \exp\left(- \frac{\left[y_i - y(x_i) \right]^2}{2\sigma_i^2} \right) \Delta y,\qquad\qquad
-\ln P(y_i|x_i) \propto \left[\frac{y_i - y(x_i)}{\sigma_i} \right]^2
$$


<br>

In principle, we could also assume  observations from other known (but non-normal) distributions, for example, discrete observations, such as $n_i$ successes and $N_i-n_i$ failures on $N_i$ trials, that sample binomial distributions with true (continuos-valued) value $N_i \pi_i$

<br>

Such situations are handled by **logistic regression**. The trick is to map probabilities $\pi\in[0,1]$ to the logit parameter $\lambda \in [-\infty, +\infty]$


$$
\lambda = \ln\left(\frac{\pi}{1-\pi} \right) \qquad \Leftrightarrow \qquad \pi = \frac{e^\lambda}{1+e^\lambda} = \frac{1}{1+ e^{-\lambda}}, \quad 1-\pi = \frac{1}{1+e^\lambda}
$$

which is assumed to be a linear function of a predictor $x_i$:

$$
\lambda_i = \alpha_0 + \alpha_1 x_i
$$

<br>

The log likelihood of a set of observations $n_i$ successes on $N_i$ trials is then
$$\begin{eqnarray}
LL(\alpha_0, \alpha_1) = \ln \prod_i  &=& \sum_i \log\left[ \binom{N_i}{n_i} \pi_i^{n_i} (1-\pi_i)^{N_i-n_i} \right] =
\\
&=&\sum_i \log\left[ \binom{N_i}{n_i} \right] + \sum_i n_i \log\left(\frac{\pi_i}{1-\pi_i}\right) + \sum_i N_i \log (1-\pi_i) =
\\
&=& \sum_i \log\left[ \binom{N_i}{n_i} \right]  + \sum_i n_i \lambda_i  -  \sum_i N_i \log (1+e^{\lambda_i})
\end{eqnarray}$$

which can be maximized by setting to zero the partial derivatives

$$
0 \stackrel{!}{=} \frac{\partial LL}{\partial a_0},\qquad\qquad 0 \stackrel{!}{=} \frac{\partial LL}{\partial a_1}
$$

<br>

We will return to logistic regression in Lecture 4.



# Pitfalls of normal assumption

The practical problem with the normal assumption is that typical observations contain far more outliers that the normal distribution predicts.

Worse, the maximum likelihood procedure attaches a weight to each observation that grows linearly with the z-score of that observation. In other words, outliers are so exceedingly unlikely that they assume an exceedingly high weight.

Maximum likelihood estimation can then disort the entire fit such as to accommodate a few outliers.

These problems are addressed by **robust statistics** (see future lectures).

# Chi-square fitting

If we assume normally distributed measurement errors, the summed error squares are a $\chi^2$-statistics

$$
 -\sum_{i=1}^N \ln P(y_i|x_i) \propto \sum_{i=1}^N\left[\frac{y_i - y(x_i; a_1 \ldots a_M)}{\sigma_i} \right]^2 = \chi^2
$$

<br>

If our model is linear in the $a$s, the distribution of $\chi^2$ values due to measurement error (i.e., the residual variation after the model parameters have been optimized), is a chi-square distribution with $N-M$ degrees of freedom, so that the p-value associated with a particular value $\chi^2$ can be obtained from the gamma distribution.

<br>

Typically these p-values are rather small (because of outliers), so that a p-value of $0.001$ might be acceptable (rather than grounds for discarding the model).

A typical value of $\chi^2$ for a moderatly good fit is $\nu=N-m$, the number of degrees of freedom.

<br>

If you do not know your measurement error $\sigma$, you can estimate from the $\chi^2$ value of the fitted model

$$
\sigma^2 \approx \frac{1}{N-M} \sum_{i=1}^N\left[y_i - y(x_i; a_1 \ldots a_M)\right]^2
$$

This allows you to assign some kind of error bar to your measurements, which reflects the 'internal consistency' of observations as judged by the fitted model.

# **5. Fitting a line (linear least squares)**

Let's consider fitting $N$ observations $(x_i,y_i)$ to a straight line

$$
y(x) = y(x;a,b) = a + bx
$$

and assume that the uncertainty $\sigma_i$ of each measurement is known.

We will assess goodness of fit with the chi-square function (summed square error)

$$
\chi^2 = \sum_{i=1}^N\left[\frac{y_i - a - bx_i}{\sigma_i} \right]^2
$$

Minimizing with respect to $a$ and $b$ gives

$$
0 = \frac{\partial\chi^2}{\partial a} = -2 \sum_i \frac{y_i-a-bx_i}{\sigma_i}
$$
$$
0 = \frac{\partial\chi^2}{\partial b} = -2 \sum_i \frac{x_i(y_i-a-bx_i)}{\sigma_i}
$$

Let's simplify the notation for what follows by defining some convenient sums

$$
S\equiv\sum_i\frac{1}{\sigma_i^2}, \qquad S_x\equiv\sum_i\frac{x_i}{\sigma_i^2}, \qquad S_y\equiv\sum_i\frac{y_i}{\sigma_i^2}
\\
S_{xx}\equiv\sum_i\frac{x_i^2}{\sigma_i^2}, \qquad S_{xy}\equiv\sum_i\frac{x_i y_i}{\sigma_i^2}
$$

With these definitions, our two equations become

$$
S_y = a S + b S_x,\qquad\qquad S_{xy} = a S_x + b S_{xx}
$$

The solution follows as

$$
a = \frac{S_{xx} S_y - S_x S_{xy}}{\Delta}, \qquad b = \frac{S S_{xy} - S_x S_y}{\Delta}, \qquad \Delta \equiv S S_{xx} - S_x S_x
$$

These are the best-fitting model parameters $a$ and $b$.

The $\chi^2$ statistic for the goodness of fit (least squared error) is

$$\begin{eqnarray}
\chi^2 = \sum_{i=1}^N\left[\frac{y_i - a - b x_i}{\sigma_i} \right]^2 &=& \sum_i \frac{y_i^2 +a^2 + b^2 x_i^2 -2a y_i -2b x_i y_i + 2a b x_i}{\sigma_i^2} =
\\
&=& S_{yy} + a^2 S + b^2 S_{xx} - 2a S_{y} -2b S_{xy} +2ab S_x
\end{eqnarray}$$


In [ ]:
from inspect import signature
# @title Fitting a line {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from scipy.stats import binom
from scipy.optimize import curve_fit


def MakeObservations( Xrange, sigmaY, rho, Nobs ):

    x_i = np.random.uniform( -Xrange, Xrange, Nobs )
    y_i = rho * x_i + sigmaY * np.random.normal( 0, 1, Nobs )

    return x_i, y_i



def plot_fitting_line(rho=0.2, Nobs=10):
    """(Slide-25)"""

    Xrange = 10;

    x = np.linspace(-Xrange,Xrange,100)

    sigmaY = 5;

    # generate observations

    (x_i, y_i) = MakeObservations( Xrange, sigmaY, rho, Nobs );

    # uniform measurement uncertainty
    sigmaY = 5;

    # prepare the sums
    S = Nobs/sigmaY**2
    Sx = np.sum(x_i/sigmaY**2)
    Sy = np.sum(y_i/sigmaY**2)
    Sxx = np.sum(x_i**2/sigmaY**2)
    Sxy = np.sum(x_i*y_i/sigmaY**2)
    Syy = np.sum(y_i**2/sigmaY**2)


    # solve for parameters of y = a + b*x
    a = (Sxx*Sy - Sx*Sxy)/(S*Sxx - Sx*Sx)
    b = (S*Sxy - Sx*Sy)/(S*Sxx - Sx*Sx)


    # chi-square statistic
    chi2 = np.sum( (y_i - a - b*x_i)**2 / sigmaY**2 );

    chi2alt = Syy + a**2*S + b**2*Sxx - 2*a*Sy - 2*b*Sxy + 2*a*b*Sx;

    est_sigma = np.sqrt( np.sum( (y_i - a - b*x_i)**2 ) / (Nobs-2) );

    # estimates sigma

    titlestring = f"Line fit: y = {a:.3f} + {b:.3f}, sigma ~ {est_sigma:.3f}"

    # Least Squares Fit from SCIPY

    # Define the model function
    def linear_model(x, a, b):
        return x * b + a


    # Fit the model
    popt, pcov = curve_fit(linear_model, x_i, y_i )


    # Plot results
    fig, axs = plt.subplots( 1, 1, figsize=(10,10))

    axs.plot( x_i, y_i, 'ko', markersize=5 )
    axs.plot( x, a+b*x, 'r', linewidth=2, label='best' )

    plt.plot(x, popt[0] + popt[1]*x, "b--", label="SCIPY")

    axs.legend(loc='upper left', fontsize=18 )
    axs.set_xlabel('x')
    axs.set_ylabel('y')
    axs.set_box_aspect(1/1)
    axs.set_title(titlestring, fontsize=24);




    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_fitting_line, rho = (-1,1,0.1), Nobs = (10, 1000, 10 )  )



interactive(children=(FloatSlider(value=0.2, description='rho', max=1.0, min=-1.0), IntSlider(value=10, descri…

<function __main__.plot_fitting_line(rho=0.2, Nobs=10)>

# Uncertainty of parameter values

To estimate the uncertainty, we need to consider the impact of measurement errors. If measurements are independent, each one will contribute additional uncertainty.  Error propagation states that the variance $\sigma_f^2$ in any function $f$ will be

$$
\sigma_f^2 = \sum_i \sigma_i^2 \left(\frac{\partial f}{\partial y_i} \right)^2
$$

For our straight line fits, we can evaluate these derivatives directly

$$
\frac{\partial S_y}{\partial y_i} = \frac{\partial}{\partial y_i} \left( \sum_i \frac{y_i}{\sigma_i^2} \right) = \frac{1}{\sigma_i^2}, \qquad\qquad \frac{\partial S_{xy}}{\partial y_i} = \frac{\partial}{\partial y_i} \left( \sum_i \frac{x_i y_i}{\sigma_i^2} \right) = \frac{x_i}{\sigma_i^2}
$$

$$
\frac{\partial a}{\partial y_i} = \frac{\partial }{\partial y_i} \left(\frac{S_{xx} S_y - S_x S_{xy}}{\Delta } \right) = \frac{S_{xx} - S_x x_i}{\sigma_i^2 \Delta}, \qquad \frac{\partial b}{\partial y_i} = \frac{\partial}{\partial y_i} \left(\frac{S S_{xy} - S_x S_y}{\Delta}\right) = \frac{S x_i - S_x}{\sigma_i^2 \Delta}
$$

Summing the squares gives us

$$
\sigma_a^2 = \sum_i \sigma_i^2 \left(\frac{S_{xx}^2 - 2 S_{xx} S_x x_i + S_x^2 x_i^2}{\sigma_i^4 \Delta^2} \right) = \frac{S_{xx}^2 S - 2 S_{xx} S_x^2 + S_x^2 S_{xx} }{\Delta^2} = \frac{S_{xx}}{\Delta}
$$
$$
\sigma_b^2 = \sum_i \sigma_i^2 \left(\frac{S^2 x_i^2 - 2 S S_x x_i + S_x^2 }{\sigma_i^4 \Delta^2} \right) = \frac{S^2 S_{xx}^2 - 2 S S_x^2 + S S_x^2 }{\Delta^2} = \frac{S}{\Delta}
$$

The covariance of $a$ and $b$ and the linear correlation coefficient are

$$
Cov(a,b) = -\frac{S_x}{\Delta}, \qquad\qquad r_{ab} = \frac{-S_x}{\sqrt{S \, S_{xx}}}
$$


# Goodness of fit

The probability that a value of chi-square as poor as the one obtained should occur by chance follows from the cumulative gamma distribution (gamdist.cdf)

$$
Q = \mathrm{gammq} \left(\frac{N-2}{2}, \frac{\chi^2}{2} \right)
$$

The relation between $\chi^2$ and the linear correlation coefficient $r$ is

$$
\chi^2 = (1-r^2) \sum_i \frac{(y_i - \bar y)^2}{\sigma_i^2}, \qquad\qquad \bar y = \frac{1}{N} \sum_i y_i
$$

# Practicalities

To minimize roundoff errors, we can implement this as follows

$$
t_i = \frac{1}{\sigma_i} \left( x_i - \frac{S_x}{S} \right), \qquad\qquad S_{tt} = \sum_i t_i^2
$$

$$
b = \frac{1}{S_{tt}} \sum_i \frac{t_i \, y_i}{\sigma_i}, \qquad\qquad a = \frac{S_y - S_x b}{S}
$$

$$
\sigma_a^2 = \frac{1}{S} \left(1 + \frac{S_x^2}{S \, S_{tt}} \right),\qquad\qquad \sigma_b^2 = \frac{1}{S_{tt}}
$$

$$
Cov(a,b) = -\frac{S_x}{S \, S_{tt}}, \qquad\qquad r_{ab} = \frac{Cov(a,b)}{\sigma_a \, \sigma_b}
$$














In [ ]:
from inspect import signature
# @title Fitting a line with nonuniform variance {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from scipy.stats import binom
from scipy.stats import gamma as gamdist
from scipy.optimize import curve_fit


def MakeObservationsNonuniform( Xrange, sigmaY, rhoY, rhoS, Nobs ):

    x_i = np.linspace( -Xrange, Xrange, Nobs )

    sigma_i = sigmaY + sigmaY * rhoS * x_i / Xrange;

    y_i = rhoY * x_i + np.random.normal( 0, sigma_i )

    return x_i, y_i, sigma_i



def plot_fitting_line_nonuniform_variance(rhoY=0.2, rhoS=0.2, Nobs=10):
    """(Slide-25)"""

    Xrange = 10;

    x = np.linspace(-Xrange,Xrange,100)

    sigmaY = 1;

    # generate observations

    (x_i, y_i, sigma_i) = MakeObservationsNonuniform( Xrange, sigmaY, rhoY, rhoS, Nobs );


    print( sigma_i[0], sigma_i[-1])


    # prepare the sums
    S    = np.sum(1/sigma_i**2)
    S_x  = np.sum(x_i/sigma_i**2)
    S_y  = np.sum(y_i/sigma_i**2)
    S_xx = np.sum(x_i**2/sigma_i**2)

    t_i  = 1/sigma_i * (x_i - S_x/S)
    S_tt = np.sum(t_i**2)


    # solve for parameters of y = a + b*x
    b = np.sum(t_i * y_i) / S_tt
    a = (S_y - S_x * b) / S

    # parameter tolerances
    sigma2_a = 1/S * (1 + S_x**2/S/S_tt)
    sigma2_b = 1/S_tt

    alow = a - 3*np.sqrt(sigma2_a)
    ahigh = a + 3*np.sqrt(sigma2_a)
    blow = b - 3*np.sqrt(sigma2_b)
    bhigh = b + 3*np.sqrt(sigma2_b)


    # Weighted Least Squares Fit

    # Define the model function
    def linear_model(x, a, b):
        return x * b + a


    # Fit the model
    popt, pcov = curve_fit(linear_model, x_i, y_i, sigma=sigma_i, absolute_sigma=True)



    # chi-square statistic
    chi2 = np.sum( (y_i - a - b*x_i)**2 / sigma_i**2 );

    p_value = 2 * (1 - gamdist.cdf(chi2, a=Nobs/2, scale=2/Nobs));

    # correlation coefficient

    rhoa = np.corrcoef(x_i, y_i)[0,1]

    yvar = np.sum( (y_i - np.mean(y_i))**2 / sigma_i**2 )


    rhob = np.sqrt( 1 - chi2 / yvar );

    est_sigma = np.sqrt( np.sum( (y_i - a - b*x_i)**2 ) / (Nobs-2) );

    # estimates sigma

    num = np.array(p_value)

    titlestring = f"Line fit: y = {a:.3f} + {b:.3f}  , p-value {num:.3f}, rho {rhoa:.2f} {rhob:.2f}"

    # Plot results
    fig, axs = plt.subplots( 1, 1, figsize=(10,10))

    axs.plot( x_i, y_i, 'ko', markersize=5, label='data' )

    axs.plot( x, alow+blow*x, color=[1.0,0.8,0.8], linewidth=1, label='3 sigma' )
    axs.plot( x, ahigh+bhigh*x, color=[1.0,0.8,0.8], linewidth=1 )
    axs.plot( x, alow+bhigh*x, color=[1.0,0.8,0.8], linewidth=1 )
    axs.plot( x, ahigh+blow*x, color=[1.0,0.8,0.8], linewidth=1 )

    axs.plot( x, a+b*x, 'r', linewidth=2, label='best' )

    axs.plot( x, rhoY*x, 'k--', linewidth=2, label='ground truth')

    plt.plot(x, popt[0] + popt[1]*x, "b-", label="SCIPY")

    axs.legend(loc='upper left', fontsize=18 )

    axs.set_xlabel('x')
    axs.set_ylabel('y')
    axs.set_box_aspect(1/1)
    axs.set_title(titlestring, fontsize=24);


    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_fitting_line_nonuniform_variance, rhoY = (-1,1,0.1), rhoS = (-0.5,0.5,0.1), Nobs = (10, 1000, 10 )  )



interactive(children=(FloatSlider(value=0.2, description='rhoY', max=1.0, min=-1.0), FloatSlider(value=0.2, de…

<function __main__.plot_fitting_line_nonuniform_variance(rhoY=0.2, rhoS=0.2, Nobs=10)>

# **6. Fitting curves (general linear least squares)**

A generalization of the above is to fit a set of data points $(x_i, y_i)$ to a linear combination of *any* functions of $x$.

For examples the functions could be powers of $x$ to obtain polynomial functions

$$
y(x) = a_0 + a_1 x + a_2 x^2 + \ldots
$$

or they could be harmonics so that we obtain Fourier series

$$
y(x) = a_0 + a_1 \cos( x / L ) + a_2 \sin( x/L ) + \ldots
$$

In general, the model is

$$
y(x) = \sum_{k=1}^M \, a_k \, X_k(x)
$$

where $X_1(x)$, $X_2(x)$, $\ldots$, $X_K(x)$ are basis functions. Of course, we must choose these basis functions to be 'orthogonal' in the sense that they should not be linear combinations of each other.

<br>

Note that the basis functions can be **highly non-linear** functions of $x$. The **linearity** of the least squares methods refers only the model's **dependence on parameters** $a_k$!

<br>

As before, we define the summed error squares as

$$
\chi^2 = \sum_{i-1}^N \left[\frac{y_i - \sum_{k=1}^M \, a_k \, X_k(x) }{\sigma_i}\right]^2
$$

and assume that $\sigma_i$ is known.  If it is not, we can start with $\sigma=1$ and estimate a better value from

$$
\sigma_\mathit{est} = \sqrt{ \frac{\chi^2}{N-M} }
$$

<br>

As before, we assume that the best parameters are those that minimize $\chi^2$. If our measurement errors were normally distributed, this would corresponds to the maximum likelihood values. Unfortunately, the assumption of normality is typically violated to a greater or lesser extent.




# Solution with normal equations

Let's construct a matrix ${\bf A} = A_{ij}$ with $N\times M$ components constructed from $M$ basis functions evaluated at $N$ abscissa values $x_i$

$$
A_{ij} = \frac{X_j(x_i)}{\sigma_i}
$$

This is called the design matrix and it must have more rows than columns, because you need (far) more data points ($N$) than unknowns ($M$).

<br>


<img src="https://raw.githubusercontent.com/jochen-braun/Advanced-Statistics/main/illu_points_and_functions.png" width="600">

<br>

In addition, we define a row vector ${\bf b}=b_i$ of length $N$ and a column vector ${\bf a} = a_j$ of length $M$ for the unknown parameters that we seek:

$$
b_i = \frac{y_i}{\sigma_i}, \qquad\qquad a_j \,\, \mathrm{unknown}
$$

<br>

Setting to zero the derivative of $\chi^2$ with respect to each parameter value $a_j$ value gives us $M$ equations

$$
0 \stackrel{!}{=} \frac{\partial \chi^2}{\partial a_j} = \sum_{i=1}^N \frac{1}{\sigma_i^2} \left[y_i - \sum_{j=1}^M \, a_j X_j(x_i) \right] \, X_k(x_i), \qquad\qquad \forall \quad k = 1, 2, \ldots M
$$

<br>

Interchanging the order of summation produces

$$
\sum_{i=1}^N \frac{y_i\, X_k(x_i)}{\sigma_i^2}  = \sum_{j=1}^M \, a_j \, \sum_{i=1}^N \frac{X_j(x_i) \, X_k(x_i)}{\sigma_i^2}
$$

<br>

Careful scrutiy reveals that the terms of this equation relate to the design matrix $\bf A$ and vector $\bf b$ defined above:

$$
\alpha_{kj} \equiv \sum_{i=1}^N \frac{X_j(x_i) \, X_k(x_i)}{\sigma_i^2} = {\bf A}^T \cdot {\bf A}
$$

$$
\beta_k \equiv \sum_{i=1}^N \frac{y_i\, X_k(x_i)}{\sigma_i^2} = {\bf A}^T \cdot {\bf b}
$$

Recalling the matrix dimensions ${\bf A}:\, (N\times M)$ and ${\bf b}:\, (N\times 1)$ makes sense of the dot products:

$$
{\bf A}^T \cdot {\bf A}: \quad (M\times M),\qquad\qquad {\bf A}^T \cdot {\bf b}: \quad (M\times 1)
$$

Our problem thus reducing to solving for $\bf a$ the **normal equations**

$$
\alpha_{kj} \cdot a_j = b_k \qquad \mathrm{or}, \,\,\mathrm{equivalently} \qquad
 \left( {\bf A}^T \cdot {\bf A} \right) \cdot {\bf a} = \left( {\bf A}^T \cdot {\bf b} \right)
$$



# Parameter uncertainties (optional)

To obtain uncertainties for the estimated parameters $\bf a$, we form the inverse matrix of $\alpha_{kj}=\left({\bf A}^T \cdot {\bf A}\right)$:

$$
C_{jk}  \equiv \left({\bf A}^T \cdot {\bf A}\right)^{-1}
$$

Left-multiplying with the inverse matrix gives

$$
\alpha_{kj} \cdot a_j = b_k \qquad \Rightarrow \qquad a_j = C_{jk} \cdot b_k
$$

or

$$
a_j = \sum_{k=1}^M C_{jk} \left[ \sum_{i=1}^N \frac{y_i\, X_k(x_i)}{\sigma_i^2}\right]
$$

As $\alpha_{jk}$ and $C_{jk}$ are independent of $y_i$, we find

$$
\frac{\partial a_j}{\partial y_i} = \sum_{k=1}^M C_{jk}  \frac{X_k(x_i)}{\sigma_i^2}
$$

and can determine the variance of $a_j$ by error propatation

$$\begin{eqnarray}
\sigma^2(a_j) &=& \sum_{i=1}^N \sigma_i^2 \left( \frac{\partial a_j}{\partial y_i} \right)^2 =
\\
\\
&=& \sum_{i=1}^N \sigma_i^2 \sum_{k=1}^M C_{jk}  \frac{X_k(x_i)}{\sigma_i^2}\sum_{l=1}^M C_{jl}  \frac{X_l(x_i)}{\sigma_i^2} =
\\
&=& \sum_{k=1}^M \sum_{l=1}^M C_{jk} C_{jl} \left[\sum_{i=1}^N \frac{X_k(x_i) X_l(x_i)}{\sigma_i^2} \right] =
\\
&=& \sum_{k=1}^M \sum_{l=1}^M C_{jk} C_{jl}  \alpha_{kl} =
\\
&=& \sum_{k=1}^M \sum_{l=1}^M C_{jk} C_{jl}  C_{kl}^{-1} = C_{jj}
\end{eqnarray}
$$

In other words, the diagonal elements of $C$ are the variances of the fitted parameters.

# Practicalities and SVD

It is not advisable to solve the normal equations directly, as this is susceptible to roundoff errors and to singularities. The deeper problem is
that least squares is both overdetermined (number of data points larger than number of parameters) and underdetermined (ambiguous parameter combinations exist).

The preferred method is singular value decomposition (SVD). In an overdetermined system, SVD produces the best compromise in a least-squares sense.  In an underdetermined system, SVD produces the solution with the smalles parameter values.

The routines implemented by Python and Matlab use this approach.

<br>

In terms of our design matrix ${\bf A}\,\, (N\times M)$ and vector ${\bf b}\,\, (N\times 1)$, the minimization can be written as follows:

$$
\min_{\bf a} \chi^2, \qquad\qquad \chi^2 = \left|{\bf A}\cdot {\bf a} - {\bf b} \right|^2
$$

The singular value decomposition of ${\bf A}\,\, (N\times M)$ is

$$
{\bf A} = {\bf U} \cdot {\bf \Sigma} \cdot {\bf V}
$$

where ${\bf U} =U_{ik} \,\,(N\times M)$ has orthogal columns ${\bf U}_{(k)}$ in the sense

$$
\sum_{i=1}^N U_{ik} U_{il} = \delta_{kl} \qquad\qquad k,l=1,2,\ldots M
$$

and ${\bf V} = V_{jk} \,\, (M\times M)$ has orthogonal columns ${\bf V}_{(k)}$ in the sense

$$
\sum_{j=1}^M V_{jk} V_{jl} = \delta_{kl} \qquad\qquad k,l=1,2,\ldots M
$$

and ${\bf \Sigma}=w_{j} \,\, (M\times M)$ is the diagonal matrix of singular values $w_j$

<br>

Solving the minimization condition for parameter vector $\bf a$ then gives

$$
{\bf a} = {\bf V} \cdot {\bf \Sigma}^{-1} \cdot {\bf U}^T \cdot {\bf b} = \sum_{j=1}^M \left(\frac{{\bf U}_{(j)}\cdot {\bf b}}{w_j} \right) {\bf V}_{(j)}
$$

So the fitted parameter values are linear combinations of the columns ${\bf V}_{(j)}$. In fact, the vectors  ${\bf V}_{(j)}$ are the principle axes of the error ellispoid of the fitted parameters ${\bf a}$.



# Parameter variance

The variance of the parameter estimates $a_j$ can be shown to be

$$
\sigma^2(a_j) = \sum_{k=1}^M \left( \frac{V_{jk}}{w_k} \right)^2
$$

The covariance is, unsurprisingly

$$
\mathrm{Cov}(a_j, a_k) = \sum_{k=1}^M \left( \frac{V_{jk}\, V_{kj}}{w_k^2} \right)
$$

See Numerical Recipes, Chapter 15.4, for details.

In [ ]:
from inspect import signature
# @title General linear least squares: Example One {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.linalg import svd
from scipy.linalg import lstsq



def MakeObservationsExampleOne( Xrange, sigmaY, rhoY, Nobs ):

    x_i = np.linspace( -25, 20, Nobs )

    y0_i = -(x_i-1)**2 + (x_i+1)**4 / 400;

    sigma_i = sigmaY + sigmaY * rhoY * x_i / Xrange;

    y_i =  y0_i + np.random.normal( 0, sigma_i )

    return x_i, y_i, y0_i, sigma_i



def plot_general_least_squares_one(sigmaY=50, rhoY=0.2, Nobs=100):
    """(Slide-25)"""

    # abscissa range
    Xrange = 25;

    x = np.linspace(-Xrange,20,100)

    # generate observations

    (x_i, y_i, y0_i, sigma_i) = MakeObservationsExampleOne( Xrange, sigmaY, rhoY, Nobs );

    # basis functions
    X_0 = np.ones(Nobs);
    X_1 = x_i;
    X_2 = x_i**2;
    X_3 = x_i**3;
    X_4 = x_i**4;

    Mbasis = 5;

    # design matrix A
    A = np.array([X_0/sigma_i, X_1/sigma_i, X_2/sigma_i, X_3/sigma_i, X_4/sigma_i]).T

    # vector b
    b = y_i/sigma_i;

    # solve for a, using scipy function lstsq
    (a, residuals, rank, s) = lstsq(A, b)


    print( "Scipy solution:", a )

    # let do this ourselves, with singular value decomposition

    (U, Sgv, Vh) = svd(A, full_matrices=True, compute_uv=True, lapack_driver='gesvd')

    # Make S into a diagonal matrix
    S  = np.zeros_like(A, dtype=float)
    np.fill_diagonal(S, Sgv)

    # V matrix
    V = Vh.T;

    # Rebuild A using U @ S @ Vh
    # A_reconstructed = U @ S @ Vh

    # make inverse matrix (linalg.inv fails mysteriously)
    S_inv  = np.zeros_like(A.T, dtype=float)
    np.fill_diagonal(S_inv, 1/Sgv)

    # solve again in our own way
    a2 = V @ S_inv @ U.T @ b

    print(" Our solution:", a2 )

    # assemble fit
    yfit_i = a2[0] + a2[1]*x_i + a2[2]*x_i**2 + a2[3]*x_i**3 + a2[4]*x_i**4;

    # parameter tolerances
    sigma2_k = np.zeros(Mbasis)
    for k in range(Mbasis):
        sigma2_k[k] = np.sum( (V[:,k]/Sgv[k])**2 )

    sigma_a = np.sqrt( sigma2_k );

    print( "Errors:", sigma_a)

    #num = np.array(p_value)

    #titlestring = f"Line fit: y = {a:.3f} + {b:.3f}  , p-value {num:.3f}, rho {rhoa:.2f} {rhob:.2f}"

    # Plot results
    fig, axs = plt.subplots( 2, 1, figsize=(10,20))

    axs[0].plot( x_i, y_i, 'ko', markersize=5, label='data' )

    axs[0].plot( x_i, y0_i, 'k--', linewidth=2, label='true' )

    axs[0].plot( x_i, yfit_i, 'r', linewidth=2, label='fit' )


    axs[0].legend(loc='upper left', fontsize=18 )

    axs[0].set_xlabel('x')
    axs[0].set_ylabel('y')
    axs[0].set_box_aspect(1/1)
    axs[0].set_title('Fitted polynomial', fontsize=24);

    labels = ['a_0', 'a_1', 'a_2', 'a_3', 'a_4']
    x = np.arange(len(a2))        # [0, 1, 2, 3]

    # Bar plot with error bars
    axs[1].bar(x, a2, yerr=sigma_a, capsize=5, color='skyblue', edgecolor='black')
    axs[1].set_xticks(x, labels)
    axs[1].set_box_aspect(1/1)
    axs[1].set_title("Parameters and tolerances")
    axs[1].set_ylabel("Value")

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_general_least_squares_one, sigmaY=(1,50,1), rhoY = (-0.5,0.5,0.1), Nobs = (100, 1000, 100 )  )



interactive(children=(IntSlider(value=50, description='sigmaY', max=50, min=1), FloatSlider(value=0.2, descrip…

<function __main__.plot_general_least_squares_one(sigmaY=50, rhoY=0.2, Nobs=100)>

In [ ]:
from inspect import signature
# @title General linear least squares: Example Two {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.linalg import svd
from scipy.linalg import lstsq



def MakeObservationsExampleTwo( Xrange, sigmaY, rhoY, Nobs ):

    Xrange = 25;
    Xperiod = 20;

    x_i = np.linspace( -25, 25, Nobs )

    y0_i = -x_i + Xperiod * np.floor( x_i / Xperiod )

    sigma_i = sigmaY + sigmaY * rhoY * x_i / Xrange;

    y_i =  y0_i + np.random.normal( 0, sigma_i )

    return x_i, y_i, y0_i, sigma_i



def plot_general_least_squares_two(sigmaY=10, rhoY=0.2, Nobs=100):
    """(Slide-25)"""

    # abscissa range
    Xrange = 25;

    x = np.linspace(-Xrange,20,100)

    # generate observations

    (x_i, y_i, y0_i, sigma_i) = MakeObservationsExampleTwo( Xrange, sigmaY, rhoY, Nobs );

    # basis functions
    X_0 = np.ones(Nobs);
    X_1 = np.cos(2*np.pi*x_i/20);
    X_2 = np.sin(2*np.pi*x_i/20);
    X_3 = np.cos(2*np.pi*x_i/10);
    X_4 = np.sin(2*np.pi*x_i/10);
    X_5 = np.cos(2*np.pi*x_i/5);
    X_6 = np.sin(2*np.pi*x_i/5);
    X_7 = np.cos(2*np.pi*x_i/2.5);
    X_8 = np.sin(2*np.pi*x_i/2.5);



    Mbasis = 9;

    # design matrix A
    A = np.array([X_0/sigma_i, X_1/sigma_i, X_2/sigma_i, X_3/sigma_i, X_4/sigma_i, X_5/sigma_i, X_6/sigma_i, X_7/sigma_i, X_8/sigma_i]).T

    # vector b
    b = y_i/sigma_i;

    # solve for a, using scipy function lstsq
    (a, residuals, rank, s) = lstsq(A, b)


    print( "Scipy solution:", a )

    # let do this ourselves, with singular value decomposition

    (U, Sgv, Vh) = svd(A, full_matrices=True, compute_uv=True, lapack_driver='gesvd')

    # Make S into a diagonal matrix
    S  = np.zeros_like(A, dtype=float)
    np.fill_diagonal(S, Sgv)

    # V matrix
    V = Vh.T;

    # Rebuild A using U @ S @ Vh
    # A_reconstructed = U @ S @ Vh

    # make inverse matrix (linalg.inv fails mysteriously)
    S_inv  = np.zeros_like(A.T, dtype=float)
    np.fill_diagonal(S_inv, 1/Sgv)

    # solve again in our own way
    a2 = V @ S_inv @ U.T @ b

    print(" Our solution:", a2 )

    # assemble fit
    yfit_i = a2[0]*X_0 + a2[1]*X_1 + a2[2]*X_2 + a2[3]*X_3 + a2[4]*X_4 + a2[5]*X_5 + a2[6]*X_6 + a2[7]*X_7 + a2[8]*X_8;

    # parameter tolerances
    sigma2_k = np.zeros(Mbasis)
    for k in range(Mbasis):
        sigma2_k[k] = np.sum( (V[:,k]/Sgv[k])**2 )

    sigma_a = np.sqrt( sigma2_k );

    print( "Errors:", sigma_a)

    #num = np.array(p_value)

    #titlestring = f"Line fit: y = {a:.3f} + {b:.3f}  , p-value {num:.3f}, rho {rhoa:.2f} {rhob:.2f}"

    # Plot results
    fig, axs = plt.subplots( 2, 1, figsize=(10,20))

    axs[0].plot( x_i, y_i, 'ko', markersize=5, label='data' )

    axs[0].plot( x_i, y0_i, 'k--', linewidth=2, label='true' )

    axs[0].plot( x_i, yfit_i, 'r', linewidth=2, label='fit' )


    axs[0].legend(loc='upper left', fontsize=18 )

    axs[0].set_xlabel('x')
    axs[0].set_ylabel('y')
    axs[0].set_box_aspect(1/1)
    axs[0].set_title('Fitted polynomial', fontsize=24);

    labels = ['a_0', 'a_1', 'a_2', 'a_3', 'a_4', 'a_5', 'a_6', 'a_7', 'a_8']
    x = np.arange(len(a2))        # [0, 1, 2, 3]

    # Bar plot with error bars
    axs[1].bar(x, a2, yerr=sigma_a, capsize=5, color='skyblue', edgecolor='black')
    axs[1].set_xticks(x, labels)
    axs[1].set_box_aspect(1/1)
    axs[1].set_title("Parameters and tolerances")
    axs[1].set_ylabel("Value")

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_general_least_squares_two, sigmaY=(1,50,1), rhoY = (-0.5,0.5,0.1), Nobs = (100, 1000, 100 )  )



interactive(children=(IntSlider(value=10, description='sigmaY', max=50, min=1), FloatSlider(value=0.2, descrip…

<function __main__.plot_general_least_squares_two(sigmaY=10, rhoY=0.2, Nobs=100)>